# AI Video Upscaler

Upscales a video from your Google Drive using Real-ESRGAN on Colab's free GPU. The result is saved back to Drive, next to the original.

**Steps**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run step 1 (tap ▶). Allow access to Google Drive when asked.
3. In step 2, type your video's path, then run it.
4. Wait for "Done". Keep this tab open while it runs.

In [ ]:
#@title 1. Setup (connect GPU + Google Drive)
import subprocess, torch
from google.colab import drive
if not torch.cuda.is_available():
    raise SystemExit("No GPU! Go to Runtime > Change runtime type > T4 GPU, then run this again.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["pip", "install", "-q", "spandrel"], check=True)
drive.mount("/content/drive")
print("Ready. Now fill in step 2.")

In [ ]:
#@title 2. Upscale
#@markdown Path to your video in Drive. A file in the main "My Drive" folder is `/content/drive/MyDrive/NAME.mp4`
INPUT = "/content/drive/MyDrive/video.mp4" #@param {type:"string"}
#@markdown How much bigger (2 = double the width and height)
SCALE = 2 #@param [2, 3, 4] {type:"raw"}
#@markdown `real-life` for camera footage, `anime` for cartoons/animation
MODEL = "real-life" #@param ["real-life", "anime"]
#@markdown Quality: lower = better and bigger file (18-24 is good)
QUALITY = 20 #@param {type:"integer"}

import json, os, shutil, subprocess, time, urllib.request
import numpy as np
import torch
from spandrel import ModelLoader

WEIGHTS = {
    "real-life": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth",
    "anime": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth",
}

if not os.path.exists(INPUT):
    # Show the videos that are in Drive so the right path can be copied into INPUT
    vids = []
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        vids += [os.path.join(root, f) for f in files if f.lower().endswith((".mov", ".mp4", ".m4v", ".mkv", ".avi"))]
        if len(vids) > 50:
            break
    msg = "Videos found in your Drive (copy one into INPUT):\n  " + "\n  ".join(vids) if vids else "No videos found in your Drive."
    if INPUT.startswith("http"):
        msg = "INPUT must be a file path, not a link.\n" + msg
    raise SystemExit(f"Can't find {INPUT}\n\n{msg}")

# --- Read video info -------------------------------------------------------
info = json.loads(subprocess.run(
    ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_streams", "-show_format",
     "-of", "json", INPUT], capture_output=True, text=True, check=True).stdout)
st = info["streams"][0]
w, h = int(st["width"]), int(st["height"])
# Phone videos are often stored sideways with a rotation flag; ffmpeg rotates on decode
rot = int(st.get("tags", {}).get("rotate", 0))
for sd in st.get("side_data_list", []):
    rot = int(sd.get("rotation", rot))
if abs(rot) % 180 == 90:
    w, h = h, w
num, den = map(int, st["avg_frame_rate"].split("/"))
fps = f"{num}/{den}" if den else "30"
duration = float(info["format"].get("duration", 0))
total = int(duration * num / den) if den else 0
out_w, out_h = int(w * SCALE) // 2 * 2, int(h * SCALE) // 2 * 2
print(f"Input: {w}x{h} @ {num/max(den,1):.2f} fps, {duration:.0f}s  ->  output {out_w}x{out_h}")

# --- Load model ------------------------------------------------------------
dev = "cuda" if torch.cuda.is_available() else "cpu"
wpath = f"/content/{MODEL}.pth" if os.path.isdir("/content") else f"{MODEL}.pth"
if not os.path.exists(wpath):
    urllib.request.urlretrieve(WEIGHTS[MODEL], wpath)
model = ModelLoader().load_from_file(wpath).to(dev).eval()
half = dev == "cuda"
if half:
    model.half()

# --- Pick encoder: GPU HEVC if it works at this size, else CPU H.264 --------
def encoder_ok(args):
    r = subprocess.run(["ffmpeg", "-v", "error", "-f", "lavfi", "-i", f"color=s={out_w}x{out_h}:d=0.1",
                        *args, "-f", "null", "-"], capture_output=True)
    return r.returncode == 0

nvenc = ["-c:v", "hevc_nvenc", "-preset", "p5", "-rc", "vbr", "-cq", str(QUALITY), "-b:v", "0", "-tag:v", "hvc1"]
x264 = ["-c:v", "libx264", "-preset", "veryfast", "-crf", str(QUALITY)]
enc = nvenc if dev == "cuda" and encoder_ok(nvenc) else x264
print("Encoder:", enc[1])

# --- Pipe: decode -> model -> encode ---------------------------------------
tmp_out = "/content/upscaled_tmp.mp4" if os.path.isdir("/content") else "upscaled_tmp.mp4"
dec = subprocess.Popen(["ffmpeg", "-v", "error", "-i", INPUT, "-vsync", "cfr", "-r", fps,
                        "-f", "rawvideo", "-pix_fmt", "rgb24", "-"], stdout=subprocess.PIPE)
encp = subprocess.Popen(["ffmpeg", "-v", "error", "-y",
                         "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{w * 4}x{h * 4}", "-r", fps, "-i", "-",
                         "-i", INPUT, "-map", "0:v", "-map", "1:a?",
                         "-vf", f"scale={out_w}:{out_h}:flags=lanczos", *enc, "-pix_fmt", "yuv420p",
                         "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", tmp_out],
                        stdin=subprocess.PIPE)

frame_bytes = w * h * 3
n, t0 = 0, time.time()
with torch.inference_mode():
    while True:
        buf = dec.stdout.read(frame_bytes)
        if len(buf) < frame_bytes:
            break
        x = torch.frombuffer(bytearray(buf), dtype=torch.uint8).to(dev).view(h, w, 3).permute(2, 0, 1)[None]
        x = x.half() / 255 if half else x.float() / 255
        y = model(x).clamp_(0, 1).mul_(255).round_().byte()[0].permute(1, 2, 0).contiguous()
        encp.stdin.write(y.cpu().numpy().tobytes())
        n += 1
        if n % 50 == 0 or n == total:
            rate = n / (time.time() - t0)
            eta = (total - n) / rate / 60 if total else 0
            print(f"\r{n}/{total} frames  {rate:.1f} fps  ~{eta:.0f} min left   ", end="", flush=True)

encp.stdin.close()
encp.wait()
dec.wait()
if encp.returncode != 0:
    raise RuntimeError("Encoding failed - see the error above.")

base, _ = os.path.splitext(INPUT)
final = f"{base}_upscaled_{SCALE}x.mp4"
shutil.move(tmp_out, final)
print(f"\nDone in {(time.time() - t0) / 60:.1f} min. Saved to: {final}")